In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@databricksgvse2e.dfs.core.windows.net/customers")
display(df)

In [0]:
df = df.drop("_rescued_data")
display(df)

In [0]:
df_domain  = df.withColumn("domain", split("email","@")[1])
display(df_domain)


In [0]:
df_name = df_domain.withColumn("full_name",concat(col('first_name'),lit(" "),col("last_name"))).drop("first_name","last_name")
display(df_name)

In [0]:
df_name.write.mode("overwrite").format("delta").save("abfss://silver@databricksgvse2e.dfs.core.windows.net/customers")

In [0]:
df = spark.read.format("delta").load("abfss://silver@databricksgvse2e.dfs.core.windows.net/customers")
display(df)

In [0]:
%sql

DESCRIBE HISTORY delta.`abfss://silver@databricksgvse2e.dfs.core.windows.net/customers`


In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_cat.silver.customers
USING DELTA
LOCATION 'abfss://silver@databricksgvse2e.dfs.core.windows.net/customers'